<a href="https://colab.research.google.com/github/jovanfranco/usdmxn-volatilidad/blob/main/clase16_series_de_tiempo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA

# 1. Ejemplos a mano
precios = np.array([20.0, 20.5, 20.3, 20.9])
cambios = np.diff(precios)
print("Cambios:", cambios.round(2), "| promedio:", round(float(cambios.mean()), 3))

phi = 0.5
print("ACF teórica de un AR(1):", [round(phi**k, 4) for k in range(1, 5)])
print("Pronóstico AR(1) a 1, 2 y 3 días con valor de hoy 2:", [round(2 * phi**k, 3) for k in (1, 2, 3)])
theta = 0.6
print("ACF(1) de un MA(1):", round(theta / (1 + theta**2), 4))
print("AIC modelo A y B:", 2*2 - 2*(-100), 2*5 - 2*(-98))
print("Rango a 10 días:", round(18.5 * (1 - 2 * 0.006 * np.sqrt(10)), 2),
      "a", round(18.5 * (1 + 2 * 0.006 * np.sqrt(10)), 2))

# 2. Simulación de un AR(1) para ver ACF y PACF
rng = np.random.default_rng(42)
n = 2000
e = rng.normal(size=n)
x = np.zeros(n)
for t in range(1, n):
    x[t] = 0.5 * x[t - 1] + e[t]
print("ACF simulada:", acf(x, nlags=4)[1:].round(3))
print("PACF simulada:", pacf(x, nlags=4)[1:].round(3))

# 3. Datos reales: USD/MXN
precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22",
                     progress=False)["Close"].squeeze()
rend = precio.pct_change().dropna()
rend100 = rend * 100

print("ADF del precio, p-valor:", round(float(adfuller(precio)[1]), 4))
print("ADF de los rendimientos, p-valor:", round(float(adfuller(rend)[1]), 4))

banda = 1.96 / np.sqrt(len(rend))
print("Banda de significancia:", round(float(banda), 3))
print("ACF de rendimientos:", acf(rend, nlags=5)[1:].round(3))
print("PACF de rendimientos:", pacf(rend, nlags=5)[1:].round(3))
print("ACF de rendimientos al cuadrado:", acf(rend**2, nlags=5)[1:].round(3))

print("Ljung Box de los rendimientos:")
print(acorr_ljungbox(rend, lags=[5, 10]).round(4))
print("Ljung Box de los rendimientos al cuadrado:")
print(acorr_ljungbox(rend**2, lags=[5, 10]).round(4))

# 4. Comparar modelos ARMA por AIC y BIC
filas = []
for p in range(3):
    for q in range(3):
        try:
            m = ARIMA(rend100.values, order=(p, 0, q)).fit()
            filas.append((p, q, m.aic, m.bic))
        except Exception:
            pass
tabla = pd.DataFrame(filas, columns=["p", "q", "AIC", "BIC"]).sort_values("BIC")
print(tabla.round(1).head())

# 5. Prueba hacia adelante: entrenar con 70%, probar con 30%
corte = int(len(rend100) * 0.7)
train, test = rend100.iloc[:corte], rend100.iloc[corte:]
m = ARIMA(train.values, order=(1, 0, 0)).fit()
const, ar1 = float(m.params[0]), float(m.params[1])
print("AR(1) estimado, constante y phi:", round(const, 4), round(ar1, 4))

previo = rend100.shift(1).loc[test.index]
pred = const + ar1 * previo
mse_modelo = float(((test - pred) ** 2).mean())
mse_base = float(((test - train.mean()) ** 2).mean())
print("R cuadrado fuera de muestra contra el promedio:", round(1 - mse_modelo / mse_base, 5))

Cambios: [ 0.5 -0.2  0.6] | promedio: 0.3
ACF teórica de un AR(1): [0.5, 0.25, 0.125, 0.0625]
Pronóstico AR(1) a 1, 2 y 3 días con valor de hoy 2: [1.0, 0.5, 0.25]
ACF(1) de un MA(1): 0.4412
AIC modelo A y B: 204 206
Rango a 10 días: 17.8 a 19.2
ACF simulada: [0.526 0.264 0.126 0.065]
PACF simulada: [ 0.527 -0.018 -0.008  0.007]


/tmp/ipykernel_3172/970376163.py:33: FutureWarning: YF.download() has changed argument auto_adjust default to True
  precio = yf.download("MXN=X", start="2020-10-07", end="2026-10-22",
/tmp/ipykernel_3172/970376163.py:38: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  print("ADF del precio, p-valor:", round(float(adfuller(precio)[1]), 4))
/tmp/ipykernel_3172/970376163.py:39: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep 

ADF del precio, p-valor: 0.1357
ADF de los rendimientos, p-valor: 0.0
Banda de significancia: 0.049
ACF de rendimientos: [-0.026 -0.018  0.007  0.04  -0.031]
PACF de rendimientos: [-0.027 -0.019  0.006  0.04  -0.028]
ACF de rendimientos al cuadrado: [0.181 0.175 0.169 0.179 0.056]
Ljung Box de los rendimientos:
    lb_stat  lb_pvalue
5    5.7006     0.3365
10  15.3242     0.1207
Ljung Box de los rendimientos al cuadrado:
     lb_stat  lb_pvalue
5   199.7353        0.0
10  211.3041        0.0
   p  q     AIC     BIC
0  0  0  3242.4  3253.1
1  0  1  3243.3  3259.4
3  1  0  3243.3  3259.4
6  2  0  3244.8  3266.2
2  0  2  3244.8  3266.3
AR(1) estimado, constante y phi: -0.0041 0.0103
R cuadrado fuera de muestra contra el promedio: -0.00298
